# EDGELEDGER AI — Model Fine-Tuning Pipeline (QLoRA / LoRA)

This notebook defines the parameter-efficient fine-tuning (PEFT / QLoRA) workflow for **EDGELEDGER AI** using the **Qwen2.5** model family.

### Pipeline Overview:
1. **Environment Setup & Verification**: PyTorch, CUDA / Apple Silicon MPS check.
2. **Dataset Compilation**: Ingests all 5 domain datasets (`macro_analysis.jsonl`, `price_action.jsonl`, `risk_management.jsonl`, `psychology.jsonl`, `trade_reviews.jsonl`).
3. **Base Model Loading**: Qwen2.5-7B-Instruct with 4-bit quantization (BitsAndBytes).
4. **LoRA Adapter Setup**: Target attention & MLP projection layers.
5. **Supervised Fine-Tuning (SFT)**: Hugging Face `trl` / `SFTTrainer`.
6. **GGUF Export**: Merge LoRA weights and export to `../model/edgeledger-trained.gguf` for local Ollama inference.

In [ ]:
# Step 1: Install & verify dependencies (Run in Google Colab / GPU Environment)
# !pip install -q torch transformers datasets peft trl bitsandbytes accelerate

In [ ]:
import os
import json
from pathlib import Path

# Set project paths relative to notebook location
NOTEBOOK_DIR = Path.cwd()
DATASET_DIR = NOTEBOOK_DIR.parent / "dataset"
MODEL_OUTPUT_DIR = NOTEBOOK_DIR.parent / "model"
VAL_FILE = NOTEBOOK_DIR / "validation.jsonl"

print(f"Dataset Directory: {DATASET_DIR.resolve()}")
print(f"Validation File:   {VAL_FILE.resolve()}")
print(f"Target Model Dir:  {MODEL_OUTPUT_DIR.resolve()}")

In [ ]:
# Step 2: Ingest all domain datasets into unified training set
dataset_files = [
    DATASET_DIR / "macro_analysis.jsonl",
    DATASET_DIR / "price_action.jsonl",
    DATASET_DIR / "risk_management.jsonl",
    DATASET_DIR / "psychology.jsonl",
    DATASET_DIR / "trade_reviews.jsonl",
]

training_samples = []
for fpath in dataset_files:
    if fpath.exists():
        with open(fpath, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    training_samples.append(json.loads(line))

print(f"Total loaded training samples: {len(training_samples)}")

In [ ]:
# Step 3: Format samples into Qwen ChatML conversational structure
def format_to_chatml(sample):
    system_prompt = sample.get("system", "You are EDGELEDGER AI, an elite trading intelligence assistant.")
    user_content = sample["instruction"]
    if sample.get("input"):
        user_content += f"\nContext: {sample['input']}"
    
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": sample["output"]},
        ]
    }

formatted_train_data = [format_to_chatml(s) for s in training_samples]
print(f"Sample preview:\n{json.dumps(formatted_train_data[0], indent=2)}")

In [ ]:
# Step 4: QLoRA & Training Hyperparameters
BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "bias": "none",
    "task_type": "CAUSAL_LM"
}

TRAIN_ARGS = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "num_train_epochs": 3,
    "warmup_ratio": 0.05,
    "logging_steps": 10,
    "output_dir": "./lora_output"
}

print("Hyperparameters configured.")

In [ ]:
# Step 5: Training Execution Cell (Gated by Safety Flag)
EXECUTE_TRAINING_NOW = False  # Set to True when ready to execute fine-tuning on GPU

if EXECUTE_TRAINING_NOW:
    print("Starting QLoRA Fine-Tuning...")
    # Training code with SFTTrainer will run here
else:
    print("Safety Guard: Training execution is disabled during project setup.")
    print("Flip EXECUTE_TRAINING_NOW = True when you are ready to train on a GPU.")

In [ ]:
# Step 6: GGUF Quantization & Ollama Export
# Export merged weights and convert to GGUF format for Ollama:
# Command:
# python llama.cpp/convert_hf_to_gguf.py ./merged_model --outfile ../model/edgeledger-trained.gguf --outtype q4_k_m
print("Target export path: ../model/edgeledger-trained.gguf")